In [ ]:
# Redirect stdout/stderr to a file for remote logging
import sys
import os

# Detect environment dynamically
env_name = "Kaggle"
base_dir = "/kaggle/working"

log_dir = base_dir
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, "log.txt")

class Logger(object):
    def __init__(self):
        self.terminal = sys.stdout
        self.log = open(log_path, "a", encoding="utf-8")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger()
sys.stderr = Logger()
print(f"📟 Logging initialized. Stderr/Stdout redirected to {log_path}")

# @title 🔑 Setup Environment & Fetch Credentials from Hub
import os
import sys
import json
import subprocess

# Check GPU capability
check_gpu_code = """
import torch
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    print(major)
else:
    print(0)
"""
try:
    res = subprocess.run(["python3", "-c", check_gpu_code], capture_output=True, text=True)
    gpu_major = int(res.stdout.strip())
    if gpu_major > 0 and gpu_major < 7:
        print(f"⚠️ Detected GPU capability {gpu_major}.0 < 7.0 (older GPU like Tesla P100). Hiding CUDA to fallback to CPU safely.")
        os.environ["CUDA_VISIBLE_DEVICES"] = ""
except Exception as e:
    print(f"Warning checking GPU capability: {e}")

print("🔄 Cloning pipeline repository...")
if os.path.exists("pipeline_code"):
    os.system("rm -rf pipeline_code")

# Step 2 repo is public as per config, clone directly
ret = os.system("git clone --quiet https://github.com/lelehoctiengtrung/lele-step2-voicestroke.git pipeline_code")
if ret != 0:
    print("❌ Failed to clone repository!")
    sys.exit(1)

sys.path.append("./pipeline_code/VPS_Steps")
os.makedirs(os.path.join(base_dir, "PROJECTS"), exist_ok=True)
os.makedirs("pipeline_code/VPS_Steps", exist_ok=True)

# Fetch credentials from Cloudflare Hub dynamically at runtime
print("📡 Fetching credentials from Cloudflare Hub...")
os.system('curl -s "https://lele-orchestrator-hub.comics2909-1.workers.dev/api/credentials?token=fbac8f27fd1833c411f62ef2225a3cc9d50b3333&file=service_account.json" -o pipeline_code/VPS_Steps/service_account.json')
os.system('curl -s "https://lele-orchestrator-hub.comics2909-1.workers.dev/api/credentials?token=fbac8f27fd1833c411f62ef2225a3cc9d50b3333&file=user_oauth2.json" -o pipeline_code/VPS_Steps/user_oauth2.json')
os.system('curl -s "https://lele-orchestrator-hub.comics2909-1.workers.dev/api/credentials?token=fbac8f27fd1833c411f62ef2225a3cc9d50b3333&file=pipeline_config.json" -o pipeline_code/VPS_Steps/pipeline_config.json')

print("✅ Environment setup complete!")
# Real-time Telegram Logger helper
def send_realtime_log():
    try:
        import sys
        sys.path.append("./pipeline_code/VPS_Steps")
        import telegram_notifier as tel
        if os.path.exists(log_path):
            with open(log_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
            log_chunk = "".join(lines[-40:])
            # Wrap in HTML pre tag for formatting
            tel.send_message(f"📟 <b>[Kaggle Session Log]</b>:\n<pre>{log_chunk}</pre>")
    except Exception as e:
        print(f"Failed to send realtime log: {e}")
send_realtime_log()


In [ ]:
# @title 📦 Install Dependencies & Setup Stroke Generator
import os
import shutil
import subprocess

print("⚙️ Installing python dependencies...")
os.system("pip install -q --timeout 120 --no-deps omnivoice && pip install -q --timeout 120 --upgrade transformers && pip install -q --timeout 120 soundfile gspread google-auth google-api-python-client requests urllib3 python-dotenv")

print("⚙️ Checking system dependencies (ffmpeg, chrome)...")
ffmpeg_available = shutil.which("ffmpeg") is not None
if not ffmpeg_available:
    print("  Installing ffmpeg...")
    os.system("apt-get update -y -q && apt-get install -y -q ffmpeg")
else:
    print("  ffmpeg is already installed.")

chrome_path = shutil.which("google-chrome-stable") or shutil.which("google-chrome") or shutil.which("chromium-browser") or "/usr/bin/google-chrome"
if os.path.exists(chrome_path) or shutil.which(chrome_path):
    print(f"  Chrome found at: {chrome_path}. Linking it...")
    os.system(f"ln -sf {chrome_path} /usr/bin/chromium-browser")
else:
    print("  Chrome not found. Downloading and installing...")
    os.system("wget -q --timeout=60 https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
    os.system("apt-get update -y -q && apt-get install -y -q ./google-chrome-stable_current_amd64.deb")
    os.system("ln -sf /usr/bin/google-chrome-stable /usr/bin/chromium-browser")
    os.system("rm -f google-chrome-stable_current_amd64.deb")

base_dir = "/kaggle/working"
local_tool_path = os.path.join(base_dir, "Chinese-Stroke-Order-Gen")
repo_tool_path = "./pipeline_code/VPS_Steps/tools/Chinese-Stroke-Order-Gen"

if os.path.exists(repo_tool_path):
    import shutil
    if os.path.exists(local_tool_path):
        shutil.rmtree(local_tool_path)
    shutil.copytree(repo_tool_path, local_tool_path)
    print("✅ Copied Chinese-Stroke-Order-Gen to working directory.")

# Adjust generator settings to use CDN
index_html = os.path.join(local_tool_path, "index.html")
if os.path.exists(index_html):
    with open(index_html, "r", encoding="utf-8") as f:
        content = f.read()
    content = content.replace("/node_modules/hanzi-writer/dist/hanzi-writer.min.js", 
                              "https://cdn.jsdelivr.net/npm/hanzi-writer@2.2/dist/hanzi-writer.min.js")
    with open(index_html, "w", encoding="utf-8") as f:
        f.write(content)

print("⚙️ Running npm install in generator folder...")
try:
    subprocess.run(["npm", "install"], cwd=local_tool_path, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=120)
    print("✅ Stroke Order Generator initialized!")
except subprocess.TimeoutExpired:
    print("⚠️ npm install timed out after 120s! Continuing anyway...")
send_realtime_log()


In [ ]:
# @title 🧠 Initialize OmniVoice Model (Preloaded from Drive & GPU Accelerated)
import os
import tarfile
import torch

# 1. Preload Model Cache from Google Drive to avoid HuggingFace download latency/hangs
local_cache_dir = "/root/.cache/huggingface/hub/models--k2-fsa--OmniVoice"
if not os.path.exists(local_cache_dir):
    os.makedirs(local_cache_dir, exist_ok=True)
    print("📥 Model cache not found. Downloading preloaded model cache from Google Drive...")
    
    model_drive_id = "19ZtD1QM7ygWoX0NQC3yP25RI7Ug-CB95"
        base_dir = "/kaggle/working"
    local_tar_path = os.path.join(base_dir, "omnivoice_model_cache.tar")
    
    # Download using Drive API to bypass warning prompt
    try:
        import sys
        sys.path.append("./pipeline_code/VPS_Steps")
        import google_api_helper as api
        from googleapiclient.http import MediaIoBaseDownload
        
        service = api.get_drive_service()
        request = service.files().get_media(fileId=model_drive_id)
        print("  Downloading tarball from Drive via API...")
        with open(local_tar_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=20*1024*1024)
            done = False
            while done is False:
                status, done = downloader.next_chunk()
                if status:
                    print(f"  Progress: {int(status.progress() * 100)}%")
        
        print("📦 Extracting model cache tarball...")
        os.system(f"tar -xf {local_tar_path} -C {local_cache_dir}")
        os.remove(local_tar_path)
        print("✅ Model cache preloaded successfully!")
    except Exception as e:
        print(f"❌ Failed to download via Drive API: {e}. Fallback to gdown...")
        # Fallback to gdown with confirm flag
        ret = os.system(f"gdown --id {model_drive_id} -O {local_tar_path} --confirm --quiet")
        if ret == 0 and os.path.exists(local_tar_path):
            print("📦 Extracting model cache tarball (gdown)...")
            os.system(f"tar -xf {local_tar_path} -C {local_cache_dir}")
            os.remove(local_tar_path)
            print("✅ Model cache preloaded successfully (gdown)!")
        else:
            print("⚠️ Fallback failed. Let HuggingFace download normally.")

# 2. Initialize OmniVoice
from omnivoice import OmniVoice

device = "cpu"
if torch.cuda.is_available():
    try:
        major, minor = torch.cuda.get_device_capability()
        if major >= 7:
            device = "cuda"
            print(f"GPU capability is {major}.{minor}. Using GPU.")
        else:
            print(f"⚠️ GPU capability is too low ({major}.{minor}) for current PyTorch. Falling back to CPU.")
    except Exception as e:
        print(f"Warning checking GPU capability: {e}. Defaulting to CPU.")
else:
    print("No GPU available. Using CPU.")

dtype = torch.float16 if device == "cuda" else torch.float32
print("📥 Initializing OmniVoice model...")
local_model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
print("✅ OmniVoice Model initialized!")


In [ ]:
# @title 🎙️ Run Step 2 Voice & Stroke Generation
try:
    import os
    import json
    import shutil
    import re
    import subprocess
    import soundfile as sf
    import urllib.parse
    import urllib.request
    import google_api_helper as api
    import telegram_notifier as tel
    from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
    
    SPREADSHEET_ID = os.environ.get("SPREADSHEET_ID", "1b6LNl7JHRiCsjK1w9VuD86GLqAfmSOtDUOm5whrGdH0")
    sample_voice_dir = "./pipeline_code/VPS_Steps/Shared/Assets/sample voice"
    PROJECTS_ROOT = "/kaggle/working/PROJECTS"
    CATEGORIES = ["hanzidegushi", "idiom", "vs_series", "dialogue", "slang"]
    
    def download_gtts(text, out_path):
        url = f"https://translate.google.com/translate_tts?ie=UTF-8&q={urllib.parse.quote(text)}&tl=zh-CN&client=tw-ob"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(out_path, 'wb') as out_file:
            out_file.write(response.read())
    
    def slow_down_audio(file_path, factor=0.8):
        if os.path.exists(file_path):
            ext = os.path.splitext(file_path)[1]
            temp_path = file_path + f".tmp{ext}"
            try:
                cmd = ["ffmpeg", "-y", "-i", file_path, "-filter:a", f"atempo={factor}", temp_path]
                subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                if os.path.exists(temp_path):
                    shutil.move(temp_path, file_path)
            except Exception as e:
                print(f"    ⚠️ Warning: Could not slow down audio using ffmpeg: {e}")
                if os.path.exists(temp_path):
                    os.remove(temp_path)
    
    def get_folder_id(url):
        if not url: return None
        url = url.strip()
        if "/" not in url and "?" not in url: return url
        if "folders/" in url:
            return url.split("folders/")[1].split("?")[0].split("/")[0]
        if "id=" in url:
            return url.split("id=")[1].split("&")[0]
        return None
    
    def download_config_from_drive(filename, parent_id, local_dest_path):
        service = api.get_drive_service()
        query = f"'{parent_id}' in parents and name = '{filename}' and trashed = false"
        results = service.files().list(q=query, fields="files(id, name)").execute()
        items = results.get('files', [])
        if not items:
            return False
        file_id = items[0]['id']
        request = service.files().get_media(fileId=file_id)
        with open(local_dest_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()
        return True
    
    def generate_stroke_order(character, local_proj_dir):
                base_dir = "/kaggle/working"
        local_tool_path = os.path.join(base_dir, "Chinese-Stroke-Order-Gen")
        print(f"✍️ Generating transparent stroke GIF for: {character}...")
        os.system(f"rm -f {local_tool_path}/output/{character}-transparent.gif")
        
        cmd = f"cd {local_tool_path} && PUPPETEER_EXECUTABLE_PATH=/usr/bin/chromium-browser HANZI_RENDER_MODE=transparent node generate.js {character}"
        os.system(cmd)
        
        generated_gif = os.path.join(local_tool_path, "output", f"{character}-transparent.gif")
        if os.path.exists(generated_gif):
            shutil.copy(generated_gif, os.path.join(local_proj_dir, "stroke_order.gif"))
            print("✅ Stroke order GIF successfully saved!")
            return True
        else:
            print("❌ Failed to generate stroke order GIF!")
            return False
    
    def process_row(sheet, category, row_to_process, character):
        local_proj_dir = os.path.join(PROJECTS_ROOT, character)
        os.makedirs(local_proj_dir, exist_ok=True)
        local_config_path = os.path.join(local_proj_dir, "config.json")
        
        rows = sheet.get_all_values()
        headers = rows[0]
        gfolder_idx = headers.index("GFolder") if "GFolder" in headers else -1
        gfolder_url = rows[row_to_process - 1][gfolder_idx] if gfolder_idx >= 0 else ""
        drive_folder_id = get_folder_id(gfolder_url)
        
        if drive_folder_id:
            print(f"📡 Downloading config.json from Drive folder ID: {drive_folder_id}...")
            success = download_config_from_drive("config.json", drive_folder_id, local_config_path)
            if not success:
                raise FileNotFoundError(f"config.json not found in GDrive folder {drive_folder_id}")
        else:
            raise ValueError(f"No GFolder URL available for row {row_to_process}")
            
        with open(local_config_path, "r", encoding="utf-8") as f:
            config = json.load(f)
            
        tel.send_telegram_notification(f"🎙️ <b>[Kaggle Step 2]</b> Đang chuyển ngữ:\n• Tab: <code>{category}</code>\n• Từ khóa: <code>{character}</code>")
        
        local_audio_dir = os.path.join(local_proj_dir, "audio")
        if os.path.exists(local_audio_dir):
            shutil.rmtree(local_audio_dir)
        os.makedirs(local_audio_dir, exist_ok=True)
        
        ref_audio = os.path.join(sample_voice_dir, "voice_preview_hien - warm and deep broadcaster.mp3")
        
        for part in config.get('voice_parts', []):
            file_path = os.path.join(local_audio_dir, part['file'])
            print(f"🔊 Processing voiceover part: {part['file']} -> '{part['text']}'")
            
            is_cloned_zh = False
            ref_zh_file = None
            instruct_zh = "female, young adult, moderate pitch"
            
            if part['type'] == 'zh_female':
                ref_zh_file = os.path.join(sample_voice_dir, "voice_preview_xiaoxi - neutral, young and friendly.mp3")
                if os.path.exists(ref_zh_file):
                    is_cloned_zh = True
            elif part['type'] == 'zh_male':
                ref_zh_file = os.path.join(sample_voice_dir, "voice_preview_haoran - deep, calm and steady.mp3")
                if os.path.exists(ref_zh_file):
                    is_cloned_zh = True
                    instruct_zh = "male, young adult, moderate pitch"
                    
            if part['type'].startswith('zh_') and not is_cloned_zh:
                print(f"  📥 Falling back to Google TTS: {part['file']}")
                download_gtts(part['text'], file_path)
                should_slow = part['type'] in ['zh_female', 'zh_male'] or any(k in part['file'] for k in ['zh_ex', 'zh_scene', 'zh_turn'])
                if should_slow:
                    slow_down_audio(file_path, factor=0.8)
            else:
                text_to_gen = part['text']
                if part['type'] == 'vi' or not part['type'].startswith('zh_'):
                    speaker_gender = "unknown"
                    match = re.search(r'vi_turn_(\d+)', part.get('file', ''))
                    if match:
                        turn_idx = match.group(1)
                        for zh_part in config.get('voice_parts', []):
                            if zh_part.get('file') == f"zh_turn_{turn_idx}.mp3":
                                if zh_part.get('type') == "zh_female":
                                    speaker_gender = "female"
                                elif zh_part.get('type') == "zh_male":
                                    speaker_gender = "male"
                                break
                    
                    if speaker_gender == "female":
                        text_to_gen = text_to_gen.replace('anh/chị', 'anh').replace('anh/ chị', 'anh').replace('anh / chị', 'anh')
                    elif speaker_gender == "male":
                        text_to_gen = text_to_gen.replace('anh/chị', 'chị').replace('anh/ chị', 'chị').replace('anh / chị', 'chị')
                    else:
                        text_to_gen = text_to_gen.replace('anh/chị', 'anh...chị').replace('anh/ chị', 'anh...chị').replace('anh / chị', 'anh...chị')
                    text_to_gen = text_to_gen.replace('/', ', ')
                
                ref = ref_zh_file if part['type'].startswith('zh_') else ref_audio
                instruct = instruct_zh if part['type'].startswith('zh_') else "female, young adult, moderate pitch"
                
                audio_output = local_model.generate(text=text_to_gen, ref_audio=ref, instruct=instruct)
                sf.write(file_path, audio_output[0], 24000)
                
                should_slow = part['type'] in ['zh_female', 'zh_male'] or any(k in part['file'] for k in ['zh_ex', 'zh_scene', 'zh_turn'])
                if should_slow:
                    slow_down_audio(file_path, factor=0.8)
                    
        print("📦 Packing audio directory into zip...")
        local_zip_path = os.path.join(local_proj_dir, "audio.zip")
        if os.path.exists(local_zip_path):
            os.remove(local_zip_path)
        subprocess.run(["zip", "-qr", "audio.zip", "audio"], cwd=local_proj_dir)
        
        voice_zip_url = "Local Only"
        if drive_folder_id:
            _, voice_zip_url = api.upload_file_to_drive(local_zip_path, "audio.zip", drive_folder_id)
            
        stroke_success = False
        if category == "hanzidegushi":
            actual_char = config.get("character", character)
            print(f"✍️ Running Chinese Stroke Order generator for '{actual_char}'...")
            stroke_success = generate_stroke_order(actual_char, local_proj_dir)
            if stroke_success:
                local_gif_path = os.path.join(local_proj_dir, "stroke_order.gif")
                if drive_folder_id:
                    api.upload_file_to_drive(local_gif_path, "stroke_order.gif", drive_folder_id)
                    
        updates = {
            "Voice": voice_zip_url,
            "Status": "Voice"
        }
        api.update_sheet_cells_batch(sheet, row_to_process, updates)
        print(f"🎉 Row {row_to_process} in tab '{category}' complete! Status set to 'Voice'.")
        
        msg = f"🎙️ <b>[Kaggle Step 2]</b> Chuyển ngữ thành công:\n• Tab: <code>{category}</code>\n• Từ khóa: <code>{character}</code>\n"
        if category == "hanzidegushi":
            msg += f"• Stroke GIF: {'✅ Đã tạo' if stroke_success else '❌ Lỗi tạo'}\n"
        msg += f"• Trạng thái -> <b>Voice</b>"
        tel.send_telegram_notification(msg)
    
    print("🎙️ Scanning tabs for pending 'Script' kịch bản...")
    processed_any = False
    
    for category in CATEGORIES:
        try:
            print(f"🔍 Scanning sheet '{category}'...")
            sheet = api.get_worksheet(SPREADSHEET_ID, category)
            rows = sheet.get_all_values()
            if not rows:
                continue
            headers = rows[0]
            status_col_idx = headers.index("Status") if "Status" in headers else -1
            char_col_idx = headers.index("Từ khoá/Chủ đề") if "Từ khoá/Chủ đề" in headers else 1
            
            if status_col_idx == -1:
                print(f"⚠️ Column 'Status' not found in sheet '{category}'. Skipping...")
                continue
                
            processed_count = 0
            for idx, row in enumerate(rows[1:], start=2):
                status_val = row[status_col_idx].strip() if len(row) > status_col_idx else ""
                if status_val.lower() == "script":
                    character = row[char_col_idx].strip()
                    print(f"🎙️ Processing Row {idx} in sheet '{category}' for character: '{character}'")
                    processed_any = True
                    process_row(sheet, category, idx, character)
                    processed_count += 1
                    
        except Exception as e:
            print(f"❌ Error processing sheet '{category}': {e}")
            tel.send_telegram_notification(f"⚠️ <b>[Kaggle Step 2]</b> Lỗi xử lý tab '{category}': {e}")
            
    if not processed_any:
        print("😴 No rows with status 'Script' found in any tab.")
        tel.send_telegram_notification("😴 <b>[Kaggle Step 2]</b> Không tìm thấy kịch bản nào cần chuyển ngữ.")
    else:
        tel.send_telegram_notification("✅ <b>[Kaggle Step 2]</b> Hoàn thành chuyển ngữ tất cả các kịch bản!")
except Exception as e:
    print(f"❌ CRITICAL EXCEPTION IN CELL 4: {e}")
    try:
        send_realtime_log()
    except Exception:
        pass
    raise e
